# Grid Strategy Backtester Notebook

This notebook uses the repo's `grid_strategy_backtester.py` directly so the notebook and script stay in sync.

Use it to:

- change strategy inputs in one cell
- run baseline vs improved backtests inline
- inspect summary, equity, and trades
- render the equity curve and diagnostics chart
- optionally export CSV and PNG outputs


## What Changed In This Version

- `normal_max_qty` now controls the transition from `NORMAL` to `CAUTION`.
- `caution_max_qty` now controls the transition from `CAUTION` to `RECOVERY`.
- `hard_max_qty` is now a strict projected inventory cap for improved mode.
- The backtest fails fast if `starting_cash` cannot fund the initial buy.
- Repair logic now removes the highest-cost lot in the internal lot book instead of using LIFO for repair sells.
- Intrabar execution is configurable with `optimistic`, `one_order_per_candle`, `conservative`, and `close_only`.


In [ ]:
from dataclasses import asdict
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

repo_root = Path.cwd()
if (repo_root / "notebooks").exists():
    repo_root = repo_root
elif (repo_root.parent / "grid_strategy_backtester.py").exists():
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from grid_strategy_backtester import (
    BacktestConfig,
    compare_baseline_vs_improved_dynamic,
    config_to_dict,
    plot_backtest_result,
    plot_diagnostics,
)


In [ ]:
# Edit this cell for each scenario.

symbol = "NSE:RELIANCE-EQ"
start = "2024-01-01"
end = "2024-01-31"
provider = "fyers"          # fyers | yfinance | yf | yahoo
resolution = "1"            # FYERS: 1, 5, 15, D | yfinance: 1 => 1m
starting_cash = None         # Example: 1_000_000
output_prefix = None

cfg = BacktestConfig(
    symbol=symbol,
    chunk_qty=70,
    initial_qty=420,
    grid_pct=0.005,
    min_profit_pct=0.006,
    fee_per_share=0.0,
    mtf_interest_annual=0.0,
    mtf_leverage=1.0,  # Example: 3.0 means interest is charged on 2/3 of inventory cost
    normal_max_qty=560,
    caution_max_qty=840,
    hard_max_qty=1050,
    recovery_extra_sell_qty=70,
    allow_repair=False,
    repair_profit_fraction=0.50,
    use_intrabar=True,
    intrabar_mode="conservative",  # optimistic | one_order_per_candle | conservative | close_only
)

display(pd.DataFrame([asdict(cfg)]))


In [ ]:
result = compare_baseline_vs_improved_dynamic(
    symbol=symbol,
    start=start,
    end=end,
    provider=provider,
    resolution=resolution,
    starting_cash=starting_cash,
    output_prefix=output_prefix,
    cfg=cfg,
)

config_df = pd.DataFrame([
    config_to_dict(
        cfg=cfg,
        starting_cash=starting_cash,
        provider=provider,
        resolution=resolution,
        start=start,
        end=end,
    )
])

print("CONFIG USED")
display(config_df.T)
print("SUMMARY")
display(result["summary"])


In [ ]:
equity_chart_path = plot_backtest_result(result)
diagnostics_chart_path = plot_diagnostics(result)

print(f"Saved equity chart: {equity_chart_path}")
print(f"Saved diagnostics chart: {diagnostics_chart_path}")


In [ ]:
print("Improved strategy trades")
display(result["improved_trades"].tail(20))

print("Improved strategy equity")
display(result["improved_equity"].tail(20))


In [ ]:
# Optional export of all CSV outputs already produced by compare_baseline_vs_improved_dynamic.
# Use this cell if you also want a config CSV from inside the notebook.

save_config_csv = False

if save_config_csv:
    config_path = f"{result['output_prefix']}_config.csv"
    config_df.to_csv(config_path, index=False)
    print(f"Saved config: {config_path}")
